In [104]:
from cassandra.cluster import Cluster
from cassandra.query import SimpleStatement, BatchStatement
from cassandra import ConsistencyLevel
import datetime

cluster = Cluster(['localhost'], port=9042)
session = cluster.connect('leaderboards')


In [105]:
import pandas as pd

## Lecturas

### Hall of Fame

In [106]:
paises = pd.read_csv('./csv_tablas/dungeons_by_country.csv')

In [107]:
paises.country.unique()

<StringArray>
['ja_JP', 'en_US', 'fr_FR', 'ko_KR', 'pt_BR', 'it_IT', 'es_ES', 'de_DE',
 'ru_RU', 'zh_CN', 'zh_TW']
Length: 11, dtype: str

In [108]:
def id_dungeons_of_country(session, country):
    query = "SELECT dungeon_id FROM dungeons_by_country WHERE country = %s;"
    statement = SimpleStatement(query, consistency_level=ConsistencyLevel.QUORUM)
    resultados = session.execute(statement, [country])
    
    dungeons_id = []

    for fila in resultados:
        dungeons_id.append(fila.dungeon_id)

    return dungeons_id

In [109]:
id_dungeons_of_country(session, 'es_ES')

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

In [110]:
def top_by_dungeon_and_country(session, country, dungeon_id, k=5):
    query = "SELECT dungeon_id, dungeon_name, time_minutes, user_name, email, date FROM hall_of_fame_by_country WHERE country = %s AND dungeon_id = %s LIMIT %s;"
    statement = SimpleStatement(query, consistency_level=ConsistencyLevel.QUORUM)
    resultados = session.execute(statement, [country, dungeon_id, k])
    
    dungeon_id = resultados[0].dungeon_id
    dungeon_name = resultados[0].dungeon_name

    top = []

    for fila in resultados:
        top.append({'email': fila.email, 'user_name': fila.user_name, 'time_minutes': fila.time_minutes, 'date': fila.date.isoformat()})

    return {'dungeon_id': dungeon_id, 'dungeon_name': dungeon_name, f'top_{k}': top}


In [111]:
top_by_dungeon_and_country(session, 'es_ES', 1, 5)

{'dungeon_id': 1,
 'dungeon_name': 'Burgstream, Culverts of the Bashful Sumo Wrestlers',
 'top_5': [{'email': 'agulloricarda@example.org',
   'user_name': 'angelino53',
   'time_minutes': 0.0,
   'date': '2012-10-08T00:00:00'},
  {'email': 'morataemperatriz@example.net',
   'user_name': 'montanareynaldo',
   'time_minutes': 0.0,
   'date': '2012-12-17T00:00:00'},
  {'email': 'eutimio76@example.net',
   'user_name': 'hervianico',
   'time_minutes': 0.0,
   'date': '2013-11-17T00:00:00'},
  {'email': 'julia03@example.net',
   'user_name': 'calistovillegas',
   'time_minutes': 0.0,
   'date': '2014-06-13T00:00:00'},
  {'email': 'felicianachaves@example.org',
   'user_name': 'yescobar',
   'time_minutes': 0.0,
   'date': '2014-08-06T00:00:00'}]}

In [112]:
def hall_of_fame(session, country):
    
    dungeon_ids = id_dungeons_of_country(session, country)

    tops_pais = []

    for dungeon_id in dungeon_ids:
        tops_pais.append(top_by_dungeon_and_country(session, country, dungeon_id, 5))
        
    return tops_pais


In [113]:
hall_of_fame(session, 'es_ES')

[{'dungeon_id': 0,
  'dungeon_name': 'Burghap, Prison of the Jealous Hippies',
  'top_5': [{'email': 'cbarrena@example.net',
    'user_name': 'cecilia28',
    'time_minutes': 0.0,
    'date': '2019-10-15T00:00:00'},
   {'email': 'begonavera@example.net',
    'user_name': 'victorino97',
    'time_minutes': 0.0,
    'date': '2019-12-17T00:00:00'},
   {'email': 'macario68@example.com',
    'user_name': 'julianesther',
    'time_minutes': 0.0,
    'date': '2022-03-25T00:00:00'},
   {'email': 'casalaaron@example.org',
    'user_name': 'ainara57',
    'time_minutes': 1.0,
    'date': '2018-05-21T00:00:00'},
   {'email': 'gisela20@example.net',
    'user_name': 'edelmiro61',
    'time_minutes': 1.0,
    'date': '2020-01-29T00:00:00'}]},
 {'dungeon_id': 1,
  'dungeon_name': 'Burgstream, Culverts of the Bashful Sumo Wrestlers',
  'top_5': [{'email': 'agulloricarda@example.org',
    'user_name': 'angelino53',
    'time_minutes': 0.0,
    'date': '2012-10-08T00:00:00'},
   {'email': 'morataempera

### User Statistics 

In [114]:
def user_statistics(session, email, dungeon_id):
    query = """SELECT time_minutes, date 
                FROM user_statistics_by_dungeon 
                WHERE email = %s AND dungeon_id = %s;"""

    statement = SimpleStatement(query, consistency_level=ConsistencyLevel.QUORUM)
    resultados = session.execute(statement, [email, dungeon_id])

    stats = []

    for fila in resultados:
        stats.append({'time_minutes': fila.time_minutes, 'date': fila.date.isoformat()})

    return stats

In [115]:
user_statistics(session, 'aabe@example.net', 0)

[{'time_minutes': 6.0, 'date': '2022-04-11T10:43:14'},
 {'time_minutes': 19.0, 'date': '2021-01-18T18:50:53'},
 {'time_minutes': 20.0, 'date': '2021-06-21T05:53:27'},
 {'time_minutes': 20.0, 'date': '2020-04-18T04:46:20'},
 {'time_minutes': 25.0, 'date': '2021-11-03T11:29:36'},
 {'time_minutes': 26.0, 'date': '2022-07-17T15:34:08'},
 {'time_minutes': 27.0, 'date': '2022-06-05T04:55:02'},
 {'time_minutes': 35.0, 'date': '2022-07-13T18:17:30'},
 {'time_minutes': 38.0, 'date': '2020-09-06T11:36:08'},
 {'time_minutes': 40.0, 'date': '2022-05-09T12:28:38'},
 {'time_minutes': 40.0, 'date': '2020-12-05T08:54:05'},
 {'time_minutes': 43.0, 'date': '2022-09-12T23:54:05'}]

In [116]:
user_statistics(session, 'aldo24@example.com', 1)

[{'time_minutes': 3.0, 'date': '2020-04-28T17:37:02'},
 {'time_minutes': 5.0, 'date': '2022-05-26T21:45:27'},
 {'time_minutes': 5.0, 'date': '2020-07-05T21:42:42'},
 {'time_minutes': 9.0, 'date': '2022-12-13T15:17:48'},
 {'time_minutes': 12.0, 'date': '2020-08-24T20:18:08'},
 {'time_minutes': 13.0, 'date': '2021-04-07T01:35:05'},
 {'time_minutes': 14.0, 'date': '2021-04-13T19:03:00'},
 {'time_minutes': 16.0, 'date': '2021-10-22T08:57:43'},
 {'time_minutes': 18.0, 'date': '2021-09-21T03:32:10'},
 {'time_minutes': 18.0, 'date': '2021-07-02T17:24:45'},
 {'time_minutes': 18.0, 'date': '2020-03-21T03:00:49'}]

### Top Horde

In [117]:
def top_horde(session, country, event_id, K):

    query = """SELECT email, user_name, n_killed
                FROM top_horde_by_event 
                WHERE country = %s AND event_id = %s 
                LIMIT %s;"""
    
    statement = SimpleStatement(query, consistency_level=ConsistencyLevel.ONE)

    resultados = session.execute(statement, [country, event_id, K])

    top = []

    for fila in resultados:
        top.append({'email': fila.email, 'user_name': fila.user_name, 'n_killed': fila.n_killed})

    return top

In [118]:
top_horde(session, 'es_ES', 5, 5)

[{'email': 'polarsenio@example.org', 'user_name': 'bbenet', 'n_killed': 30},
 {'email': 'jose-franciscoluis@example.org',
  'user_name': 'maciasmaria-jose',
  'n_killed': 27},
 {'email': 'yagoquintanilla@example.com',
  'user_name': 'nievesverdu',
  'n_killed': 27},
 {'email': 'aguirrerico@example.org',
  'user_name': 'villarmartin',
  'n_killed': 26},
 {'email': 'leandramorata@example.com',
  'user_name': 'jacinta30',
  'n_killed': 26}]

## Escritura

### User finish dungeon

In [119]:
from cassandra.query import BatchStatement
from cassandra import ConsistencyLevel

def user_finish_dungeon(session, country, dungeon_id, time_minutes, email, user_name, date, dungeon_name):
    # Usamos QUORUM para asegurar que la lectura/escritura sea consistente en el clúster
    batch = BatchStatement(consistency_level=ConsistencyLevel.QUORUM)
    
    # Registrar la combinación país-mazmorra
    insert_dungeons_by_country = """
        INSERT INTO dungeons_by_country 
        (country, dungeon_id) 
        VALUES (%s, %s)
    """
    
    # Obtener el récord anterior (tiempo y fecha) desde la tabla de usuario
    get_previous_record = """
        SELECT time_minutes, date 
        FROM user_statistics_by_dungeon 
        WHERE email = %s AND dungeon_id = %s
        ORDER BY time_minutes ASC LIMIT 1
    """

    # Ejecutamos la consulta pasándole (email, dungeon_id)
    previous_record = session.execute(get_previous_record, (email, dungeon_id)).one()
    insertar_nuevo_record = False

    # Preparar la inserción del nuevo récord en el Hall of Fame
    insert_hall_of_fame = """
        INSERT INTO hall_of_fame_by_country 
        (country, dungeon_id, time_minutes, email, user_name, date, dungeon_name) 
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    """

    # Lógica para determinar si el tiempo actual es un nuevo récord
    if previous_record is not None:
        if time_minutes < previous_record.time_minutes:
            print(f"Nuevo récord para {user_name} en {dungeon_name} ({country}): {time_minutes} minutos (anterior: {previous_record.time_minutes} minutos)")
            insertar_nuevo_record = True
        else:
            print(f"{user_name} ha completado {dungeon_name} en {time_minutes} minutos, pero no ha superado su récord anterior de {previous_record.time_minutes} minutos.")
    else:
        print(f"Primer récord para {user_name} en {dungeon_name} ({country}): {time_minutes} minutos")
        insertar_nuevo_record = True

    # Preparar la inserción en el historial personal del jugador
    insert_user_stats = """
        INSERT INTO user_statistics_by_dungeon 
        (email, dungeon_id, time_minutes, date) 
        VALUES (%s, %s, %s, %s)
    """

    # Preparar el borrado del récord viejo (¡Ahora con la fecha incluida!)
    delete_previous_record = """
        DELETE FROM hall_of_fame_by_country 
        WHERE country = %s AND dungeon_id = %s AND time_minutes = %s AND date = %s AND email = %s
    """
    
    # Construcción del batch de operaciones
    batch.add(insert_dungeons_by_country, (country, dungeon_id))

    if insertar_nuevo_record:   
        # Añadimos el nuevo súper tiempo
        batch.add(insert_hall_of_fame, (country, dungeon_id, time_minutes, email, user_name, date, dungeon_name))
        
        # Si había un récord previo, lo borramos usando la ruta exacta: tiempo + fecha + email
        if previous_record is not None:
            batch.add(delete_previous_record, (country, dungeon_id, previous_record.time_minutes, previous_record.date, email))
            
    # Registramos las estadísticas (esto se ejecuta siempre, haya récord o no)
    batch.add(insert_user_stats, (email, dungeon_id, time_minutes, date))
    
    # Ejecutamos el batch de forma atómica
    session.execute(batch)
    print(f"Intento de {user_name} procesado y guardado con éxito en las tablas.")

In [120]:
user_finish_dungeon(
    session=session,
    country='ES',
    dungeon_id=101,
    time_minutes=11,
    email='player@email.com',
    user_name='Thor',
    date="2026-03-23T22:54:34.450000",
    dungeon_name='Cueva Helada'
)

Primer récord para Thor en Cueva Helada (ES): 11 minutos
Intento de Thor procesado y guardado con éxito en las tablas.


In [121]:
user_statistics(session, 'player@email.com', 101)

[{'time_minutes': 11.0, 'date': '2026-03-23T22:54:34.450000'}]

In [122]:
hall_of_fame(session, 'ES')

[{'dungeon_id': 101,
  'dungeon_name': 'Cueva Helada',
  'top_5': [{'email': 'player@email.com',
    'user_name': 'Thor',
    'time_minutes': 11.0,
    'date': '2026-03-23T22:54:34.450000'}]}]

In [123]:
user_finish_dungeon(
    session=session,
    country='ES',
    dungeon_id=101,
    time_minutes=3,
    email='player@email.com',
    user_name='Thor',
    date="2026-03-23T22:54:34.450000",
    dungeon_name='Cueva Helada'
)

Nuevo récord para Thor en Cueva Helada (ES): 3 minutos (anterior: 11.0 minutos)
Intento de Thor procesado y guardado con éxito en las tablas.


In [124]:
hall_of_fame(session, 'ES')

[{'dungeon_id': 101,
  'dungeon_name': 'Cueva Helada',
  'top_5': [{'email': 'player@email.com',
    'user_name': 'Thor',
    'time_minutes': 3.0,
    'date': '2026-03-23T22:54:34.450000'}]}]

### User kills monster during Horde event

In [125]:
def user_kills_monster(session, country, event_id, email, user_name):
    # Paso 1: Leer n_killed actual del jugador (usa el índice secundario sobre email)
    get_kills_query = """
        SELECT n_killed
        FROM top_horde_by_event
        WHERE country = %s AND event_id = %s AND email = %s;
    """
    statement = SimpleStatement(get_kills_query, consistency_level=ConsistencyLevel.ONE)
    row = session.execute(statement, [country, event_id, email]).one()

    # Paso 2: Calcular nuevo n_killed
    n_killed_actual = row.n_killed if row else 0
    n_killed_total = n_killed_actual + 1

    # Paso 3: DELETE + INSERT en BATCH
    batch = BatchStatement(consistency_level=ConsistencyLevel.ONE)

    # Borrar fila anterior si existe
    if row:
        delete_query = """
            DELETE FROM top_horde_by_event
            WHERE country = %s AND event_id = %s AND n_killed = %s AND email = %s;
        """
        batch.add(delete_query, (country, event_id, n_killed_actual, email))

    # Insertar fila actualizada
    insert_query = """
        INSERT INTO top_horde_by_event
        (country, event_id, n_killed, email, user_name)
        VALUES (%s, %s, %s, %s, %s)
    """
    batch.add(insert_query, (country, event_id, n_killed_total, email, user_name))

    # Ejecutar cambios
    try:
        session.execute(batch)
        print(f"Récord de {user_name} actualizado. Total bajas: {n_killed_total}.")
    except Exception as e:
        print(f"Error al actualizar el Leaderboard: {e}")


**Nota**: Aunque el índice secundario soluciona este problema para la práctica, en sistemas masivos a nivel de empresas como Netflix o Discord, a veces prefieren crear dos tablas separadas: una tabla rápida para buscar al usuario por email y otra tabla ordenada para el leaderboard. Pero para este prototipo, el índice secundario cumple perfectamente el objetivo sin complicar el modelo de datos.

Como ejemplo vamos a modificar la siguiente fila: 

de_DE,0,7,trommleremilie,kabuskarin@example.net

In [126]:
user_kills_monster(
    session=session,
    country='de_DE',
    event_id=0,
    email='kabuskarin@example.net',
    user_name='trommleremilie'
)

Récord de trommleremilie actualizado. Total bajas: 8.
